In [16]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


Dataset with state is considered for the analysis of RQ1.

In [17]:
yougov = pd.read_csv("yougov_oxcgrt_rolling_cleaned_preprocessed_withstate.csv") 

print(yougov.columns)
print(f"\nyougov columns:",len(yougov.columns))

Index(['RecordNo', 'Date', 'Non-household contacts', 'age', 'household_size',
       'Wellbeing', 'Perceived Severity', 'Perceived Susceptibility',
       'face_mask_scale', 'face_mask_binary',
       'general_protective_behavior_scale',
       'general_protective_behavior_binary',
       'protective_behavior_nomask_scale', 'week', '7days_rolling_cases',
       '7days_rolling_deaths', 'mandate_start_date', 'mandate_period',
       'Isolate if unwell_Not sure', 'Isolate if unwell_Yes',
       'Isolate if instructed_Not sure',
       'Isolate if instructed_Somewhat unwilling',
       'Isolate if instructed_Somewhat willing',
       'Isolate if instructed_Very unwilling',
       'Isolate if instructed_Very willing', 'gender_Male',
       'state_New South Wales', 'state_Northern Territory', 'state_Queensland',
       'state_South Australia', 'state_Tasmania', 'state_Victoria',
       'state_Western Australia', 'employment_status_Not working',
       'employment_status_Part time employment'

30 columns have been extended to 68 columns after encoding.

RQ1 ---------------------------------------------------------------

Create before mandates and after mandates datasets

In [18]:
# before and after mandates 

print("Final total observations: ", len(yougov))

print("Observations before mandates: ", (yougov['mandate_period']==0).sum())
print("Observations after mandates: ", (yougov['mandate_period']==1).sum())

Final total observations:  39890
Observations before mandates:  14842
Observations after mandates:  25048


Before mandates: 14,842 (around 37.2%) After mandates: 25,048 (around 62.8%). These are enough observations for RF and XGBoost.

In [19]:
yougov.isna().value_counts()

RecordNo  Date   Non-household contacts  age    household_size  Wellbeing  Perceived Severity  Perceived Susceptibility  face_mask_scale  face_mask_binary  general_protective_behavior_scale  general_protective_behavior_binary  protective_behavior_nomask_scale  week   7days_rolling_cases  7days_rolling_deaths  mandate_start_date  mandate_period  Isolate if unwell_Not sure  Isolate if unwell_Yes  Isolate if instructed_Not sure  Isolate if instructed_Somewhat unwilling  Isolate if instructed_Somewhat willing  Isolate if instructed_Very unwilling  Isolate if instructed_Very willing  gender_Male  state_New South Wales  state_Northern Territory  state_Queensland  state_South Australia  state_Tasmania  state_Victoria  state_Western Australia  employment_status_Not working  employment_status_Part time employment  employment_status_Retired  employment_status_Unemployed  Confidence in Goverment's response_A lot of confidence  Confidence in Goverment's response_Don't know  Confidence in Goverment

No missing values.

Data Splitting 

before and after mandate datasets -  face mask wearing

In [20]:
# Data splitting 80-20% for before and after mandate datasets - face mask 

from sklearn.model_selection import train_test_split

# split data set 
before_mandates= yougov[yougov['mandate_period']==0] # rows
after_mandates= yougov[yougov['mandate_period']==1]

# 80% Train  20% Test for both before and after groups
before_train_facemask, before_test_facemask = train_test_split(
    before_mandates,test_size=0.20,random_state=42,stratify=before_mandates['face_mask_binary']
)

after_train_facemask, after_test_facemask = train_test_split(
    after_mandates,test_size=0.20,random_state=42, stratify=after_mandates['face_mask_binary']
)

print("Before mandates - facemask:")
print("Training:", len(before_train_facemask))
print("Test:", len(before_test_facemask))

print("\nAfter mandates - facemask:")
print("Training:", len(after_train_facemask))
print("Test:", len(after_test_facemask))

before_train_facemask.to_csv("before_train_facemask.csv", index=False)
before_test_facemask.to_csv("before_test_facemask.csv", index=False)

after_train_facemask.to_csv("after_train_facemask.csv", index=False)
after_test_facemask.to_csv("after_test_facemask.csv", index=False)



Before mandates - facemask:
Training: 11873
Test: 2969

After mandates - facemask:
Training: 20038
Test: 5010


before and after mandate datasets -  general protective behaviours

In [ ]:
# Data splitting 80-20% for before and after mandate datasets - general protective behaviours

# split data set 

# 80% Train  20% Test 
before_train_general_behav, before_test_general_behav = train_test_split(
    before_mandates,test_size=0.20,random_state=42,stratify=before_mandates['general_protective_behavior_binary']
)

after_train_general_behav, after_test_general_behav = train_test_split(
    after_mandates,test_size=0.20,random_state=42, stratify=after_mandates['general_protective_behavior_binary']
)

print("Before mandates - general_behav :")
print("Training:", len(before_train_general_behav))
print("Test:", len(before_test_general_behav))

print("\nAfter mandates - general_behav:")
print("Training:", len(after_train_general_behav))
print("Test:", len(after_test_general_behav))

before_train_general_behav.to_csv("before_train_general_behav.csv", index=False)
before_test_general_behav.to_csv("before_test_general_behav.csv", index=False)

after_train_general_behav.to_csv("after_train_general_behav.csv", index=False)
after_test_general_behav.to_csv("after_test_general_behav.csv", index=False)



Before mandates - general_behav :
Training: 11873
Test: 2969

After mandates - general_behav:
Training: 20038
Test: 5010


Check Target Class Imbalance

In [22]:
print('\nAll sets:')
print('\nBefore Mandates: ',before_mandates['face_mask_binary'].value_counts(normalize=True)*100)
print('\nAfter Mandates: ',after_mandates['face_mask_binary'].value_counts(normalize=True)*100)

print('\nBefore Mandates: ',before_mandates['general_protective_behavior_binary']
      .value_counts(normalize=True)*100)
print('\nAfter Mandates: ',after_mandates['general_protective_behavior_binary']
      .value_counts(normalize=True)*100)



print('\nTrain sets:')

print('\nBefore Mandates train set: ',before_train_facemask['face_mask_binary'].value_counts(normalize=True)*100)
print('\nAfter Mandates train set: ',after_train_facemask['face_mask_binary'].value_counts(normalize=True)*100)

print('\nBefore Mandates: ',before_train_general_behav['general_protective_behavior_binary'].value_counts(normalize=True)*100)
print('\nAfter Mandates: ',after_train_general_behav['general_protective_behavior_binary'].value_counts(normalize=True)*100)



print('\nTest sets:')

print('\nBefore Mandates test set: ',before_test_facemask['face_mask_binary'].value_counts(normalize=True)*100)
print('\nAfter Mandates test set: ',after_test_facemask['face_mask_binary'].value_counts(normalize=True)*100)

print('\nBefore Mandates: ',before_test_general_behav['general_protective_behavior_binary'].value_counts(normalize=True)*100)
print('\nAfter Mandates: ',after_test_general_behav['general_protective_behavior_binary'].value_counts(normalize=True)*100)



All sets:

Before Mandates:  face_mask_binary
0    74.248754
1    25.751246
Name: proportion, dtype: float64

After Mandates:  face_mask_binary
1    70.712233
0    29.287767
Name: proportion, dtype: float64

Before Mandates:  general_protective_behavior_binary
1    52.863495
0    47.136505
Name: proportion, dtype: float64

After Mandates:  general_protective_behavior_binary
1    72.440913
0    27.559087
Name: proportion, dtype: float64

Train sets:

Before Mandates train set:  face_mask_binary
0    74.252506
1    25.747494
Name: proportion, dtype: float64

After Mandates train set:  face_mask_binary
1    70.71065
0    29.28935
Name: proportion, dtype: float64

Before Mandates:  general_protective_behavior_binary
1    52.859429
0    47.140571
Name: proportion, dtype: float64

After Mandates:  general_protective_behavior_binary
1    72.44236
0    27.55764
Name: proportion, dtype: float64

Test sets:

Before Mandates test set:  face_mask_binary
0    74.233749
1    25.766251
Name: proport